# Distributed Inference

## Learning Objectives
1. Simulate tensor parallelism by splitting weight matrices across multiple "devices" and measuring throughput
2. Implement pipeline parallelism with micro-batching and quantify bubble overhead
3. Build a continuous batching system with dynamic queue management and fault-tolerant rerouting
4. Compare single-GPU, tensor-parallel, and pipeline-parallel configurations using throughput and latency metrics


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
import threading
import queue
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')


## Level 1: Basic Tensor Parallelism Simulation

Split a weight matrix column-wise across two simulated devices using numpy arrays. Each 'device' computes a partial matrix-multiply, then results are concatenated (allgather). Compare output to single-device reference to verify correctness, then benchmark throughput.


In [ ]:
# Level 1: Tensor Parallelism — split weight matrix across 2 simulated devices
# In real TP, each GPU holds num_heads/TP_degree attention heads
# Here we simulate by splitting a weight matrix column-wise across 2 'devices'

def tensor_parallel_matmul(
    x: np.ndarray,          # Input: (batch, in_features)
    W: np.ndarray,          # Full weight: (in_features, out_features)
    tp_degree: int = 2,     # Number of simulated devices
) -> Tuple[np.ndarray, float, float]:
    """Simulate tensor-parallel matmul: split W column-wise, compute partial outputs, allgather."""
    # Split weight matrix column-wise across TP devices
    out_features = W.shape[1]
    chunk_size = out_features // tp_degree
    W_shards = [W[:, i * chunk_size:(i + 1) * chunk_size] for i in range(tp_degree)]

    t0 = time.perf_counter()
    # Each device computes its partial output (simulated in sequence here)
    partial_outputs = [x @ W_shard for W_shard in W_shards]   # (batch, chunk_size) each
    # AllGather: concatenate partial outputs along out_features dimension
    output = np.concatenate(partial_outputs, axis=1)          # (batch, out_features)
    tp_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    single_output = x @ W                                      # Single-device reference
    single_time = time.perf_counter() - t0

    return output, tp_time, single_time


# Setup: realistic dimensions — hidden=1024, batch=32
batch_size = 32
in_features = 1024
out_features = 4096     # Typical FFN intermediate size: 4x hidden

x_input = np.random.randn(batch_size, in_features).astype(np.float32)
W_full  = np.random.randn(in_features, out_features).astype(np.float32) * 0.02

# Run tensor parallel simulation with TP degrees 1, 2, 4
print(f'Input shape: {x_input.shape}, Weight shape: {W_full.shape}')
print('=' * 55)

results = {}
for tp in [1, 2, 4]:
    if out_features % tp != 0:
        continue
    out, tp_t, single_t = tensor_parallel_matmul(x_input, W_full, tp_degree=tp)
    ref_out = x_input @ W_full
    max_err = np.max(np.abs(out - ref_out))
    results[tp] = {'time': tp_t, 'error': max_err}
    print(f'TP={tp}: output_shape={out.shape}, max_error={max_err:.2e}, tp_time={tp_t*1000:.3f}ms')

print('\n--- Single-device baseline ---')
_, _, single_t = tensor_parallel_matmul(x_input, W_full, tp_degree=1)
print(f'Single device: {single_t*1000:.3f}ms')

# Bandwidth calculation: AllReduce cost at TP=4
# Cost = 2 * hidden_size * batch * 4 bytes * (TP-1)/TP per layer
allreduce_bytes = 2 * in_features * batch_size * 4 * (4 - 1) / 4
print(f'\nAllReduce data volume (TP=4, 1 layer): {allreduce_bytes / 1024:.1f} KB')
print('Real NVLink bandwidth: ~600 GB/s  ->  negligible overhead')
print('Real InfiniBand bandwidth: ~25 GB/s -> significant overhead, avoid cross-node TP')


## Level 2: Pipeline Parallelism with Micro-Batching

Simulate a 4-stage transformer pipeline using PyTorch nn.Linear layers. Each stage processes one micro-batch and passes activations to the next stage. Measure bubble overhead at different micro-batch counts and show throughput improvement when bubble is filled. Includes OOM error handling and timing per stage.


In [ ]:
# Level 2: Pipeline Parallelism — 4 stages, micro-batching to reduce bubble overhead
# Real pipeline parallelism sends activations between GPUs; here we simulate with CPU stages

class TransformerStage(nn.Module):
    """One stage of a pipeline-parallel transformer (subset of layers)."""
    def __init__(self, hidden: int, num_layers: int, stage_id: int):
        super().__init__()
        self.stage_id = stage_id
        # Each stage has num_layers transformer-FFN blocks
        layers = []
        for _ in range(num_layers):
            layers += [
                nn.Linear(hidden, 4 * hidden),
                nn.GELU(),
                nn.Linear(4 * hidden, hidden),
                nn.LayerNorm(hidden),
            ]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def run_pipeline(
    stages: List[TransformerStage],
    macro_batch: torch.Tensor,   # (total_batch, seq_len, hidden)
    num_micro_batches: int,
) -> Tuple[torch.Tensor, float, float]:
    """Simulate pipeline execution with micro-batching. Return output + timing stats."""
    total_batch = macro_batch.shape[0]
    micro_size = total_batch // num_micro_batches
    if total_batch % num_micro_batches != 0:
        raise ValueError(f'total_batch={total_batch} not divisible by num_micro_batches={num_micro_batches}')

    # Split macro batch into micro batches
    micro_batches = macro_batch.split(micro_size, dim=0)
    outputs = []

    # Simulate 1F1B schedule: forward all micro-batches through all stages
    # In real PP, stages overlap; here we measure total sequential time as proxy
    stage_times = defaultdict(float)

    try:
        pipeline_start = time.perf_counter()
        for mb in micro_batches:
            x = mb
            for stage in stages:
                t0 = time.perf_counter()
                with torch.no_grad():
                    x = stage(x.view(x.size(0) * x.size(1), -1))   # flatten seq
                    x = x.view(mb.size(0), mb.size(1), -1)          # restore
                stage_times[stage.stage_id] += time.perf_counter() - t0
            outputs.append(x)
        total_time = time.perf_counter() - pipeline_start
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'OOM in pipeline stage: reduce micro-batch size or hidden dim')
            raise
        raise

    full_output = torch.cat(outputs, dim=0)
    # Bubble overhead approximation: (P-1) / (M + P - 1)
    P = len(stages)
    M = num_micro_batches
    bubble_fraction = (P - 1) / (M + P - 1)
    return full_output, total_time, bubble_fraction


# Setup: 4-stage pipeline, hidden=256 (small for CPU demo), seq_len=64
HIDDEN = 256
SEQ_LEN = 64
TOTAL_BATCH = 32
LAYERS_PER_STAGE = 2   # 2 transformer blocks per stage, 8 total
NUM_PP_STAGES = 4

stages = [TransformerStage(HIDDEN, LAYERS_PER_STAGE, stage_id=i) for i in range(NUM_PP_STAGES)]
for s in stages:
    s.to(device).eval()

macro_input = torch.randn(TOTAL_BATCH, SEQ_LEN, HIDDEN, device=device)

print(f'Pipeline: {NUM_PP_STAGES} stages x {LAYERS_PER_STAGE} layers/stage = {NUM_PP_STAGES*LAYERS_PER_STAGE} total layers')
print(f'Input: batch={TOTAL_BATCH}, seq_len={SEQ_LEN}, hidden={HIDDEN}')
print('=' * 55)

# Run with different micro-batch counts to show bubble reduction
mb_results = {}
for M in [1, 2, 4, 8, 16]:
    if TOTAL_BATCH % M != 0:
        continue
    output, elapsed, bubble = run_pipeline(stages, macro_input, num_micro_batches=M)
    mb_results[M] = {'time': elapsed, 'bubble': bubble}
    throughput = TOTAL_BATCH / elapsed
    print(f'M={M:2d} micro-batches: bubble={bubble*100:.1f}%, '
          f'elapsed={elapsed*1000:.0f}ms, '
          f'throughput={throughput:.0f} samples/s, '
          f'output_shape={output.shape}')

print('\nKey insight: at M=1, bubble dominates (75% for PP=4). At M=16, bubble drops to ~16%.')


# --- Allreduce communication overhead simulation ---
# In real distributed systems, allreduce latency bounds scaling efficiency
# Modeled as: latency = alpha + beta * message_size_bytes

ALPHA_NS = 1_000     # startup latency: 1 microsecond per message
BETA_GBPS = 200.0    # InfiniBand bandwidth: 200 Gb/s = 25 GB/s

def allreduce_latency_ms(
    tensor_numel: int,
    dtype_bytes: int = 4,
    num_gpus: int = 4,
    bandwidth_gbps: float = BETA_GBPS,
) -> float:
    """Estimate allreduce latency using ring-allreduce formula.

    Ring-allreduce: each GPU sends and receives (num_gpus-1) chunks.
    Total data per GPU = 2 * (num_gpus-1)/num_gpus * message_bytes.
    """
    msg_bytes = tensor_numel * dtype_bytes
    # Ring-allreduce efficient data transfer per GPU
    transfer_bytes = 2.0 * (num_gpus - 1) / num_gpus * msg_bytes
    bw_bytes_per_ms = bandwidth_gbps * 1e9 / 8 / 1e3  # bytes per ms
    return transfer_bytes / bw_bytes_per_ms + ALPHA_NS / 1e6  # in ms


# Simulate scaling efficiency across GPU counts
# Hidden=4096, intermediate=16384 (LLaMA-13B FFN), batch=32
TENSOR_NUMEL = 4096 * 4   # one row of hidden -> comm per layer
print()
print('Allreduce latency and scaling efficiency (simulated, IB 200Gb/s):')
print(f'{"Num GPUs":10s} {"AR Latency":12s} {"Compute":12s} {"Efficiency":12s} {"Speedup":10s}')
print('-' * 58)

single_compute_ms = 5.0   # Simulated forward pass time on single GPU
ar_results = {}
for n_gpus in [1, 2, 4, 8, 16]:
    ar_lat = allreduce_latency_ms(TENSOR_NUMEL, num_gpus=n_gpus) if n_gpus > 1 else 0.0
    # With TP, compute divides by n_gpus but allreduce adds overhead each layer
    compute_ms = single_compute_ms / n_gpus
    total_ms = compute_ms + ar_lat
    efficiency = (single_compute_ms / n_gpus) / total_ms  # ideal vs actual
    speedup = single_compute_ms / total_ms
    ar_results[n_gpus] = efficiency
    print(f'{n_gpus:10d} {ar_lat:10.3f}ms {compute_ms:10.3f}ms {efficiency:12.1%} {speedup:10.2f}x')

print()
print('Scaling efficiency = compute_fraction / total_time')
print('Drops below 80% around TP=8 on IB; use NVLink (600GB/s) to maintain >95%')

# Gradient sync overhead for distributed training (shown for contrast)
print()
print('Micro-batch timing breakdown (from run_pipeline above):')
if mb_results:
    baseline_time = mb_results.get(1, {}).get('time', None)
    for M, res in sorted(mb_results.items()):
        eff = (baseline_time / res['time'] * 1 / M) if baseline_time else 0.0
        print(f'  M={M:2d}: bubble={res["bubble"]*100:.1f}%, '
              f'time={res["time"]*1000:.1f}ms, '
              f'device_eff={1-res["bubble"]:.2f}')


## Real-World Example 1: Tensor Parallelism for Large Linear Layers

Simulate column-parallel and row-parallel linear layers as used in tensor-parallel attention (Megatron-LM style). Split the weight column-wise for the first linear (parallel GEMM), compute partial outputs, then row-wise for the second linear followed by AllReduce. Measure memory per device and speedup vs single device at different TP degrees.


In [ ]:
# Real-World Example 1: Column-parallel + Row-parallel linear (Megatron-LM style)
# Column-parallel: split W by columns -> each GPU produces partial output
# Row-parallel: split W by rows -> each GPU gets partial input, reduce after

class ColumnParallelLinear(nn.Module):
    """First linear in FFN: split output features across TP devices."""
    def __init__(self, in_f: int, out_f: int, tp_degree: int):
        super().__init__()
        assert out_f % tp_degree == 0, 'out_features must be divisible by tp_degree'
        self.tp_degree = tp_degree
        self.chunk_size = out_f // tp_degree
        # In real TP each device holds only its shard; here we hold all for simulation
        self.weight_shards = nn.ParameterList([
            nn.Parameter(torch.randn(in_f, self.chunk_size) * 0.02)
            for _ in range(tp_degree)
        ])

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        """Returns list of partial outputs, one per simulated device."""
        return [x @ shard for shard in self.weight_shards]


class RowParallelLinear(nn.Module):
    """Second linear in FFN: each device computes partial contribution, then AllReduce."""
    def __init__(self, in_f: int, out_f: int, tp_degree: int):
        super().__init__()
        assert in_f % tp_degree == 0
        self.tp_degree = tp_degree
        chunk_in = in_f // tp_degree
        self.weight_shards = nn.ParameterList([
            nn.Parameter(torch.randn(chunk_in, out_f) * 0.02)
            for _ in range(tp_degree)
        ])

    def forward(self, partial_inputs: List[torch.Tensor]) -> torch.Tensor:
        """AllReduce: sum partial contributions from each device."""
        partials = [inp @ shard for inp, shard in zip(partial_inputs, self.weight_shards)]
        return torch.stack(partials, dim=0).sum(dim=0)   # AllReduce (sum)


# Benchmark TP degrees for FFN block (hidden=512, intermediate=2048)
HIDDEN = 512
INTERMEDIATE = 2048
BATCH = 16

x_ff = torch.randn(BATCH, HIDDEN, device=device)

print(f'FFN block: hidden={HIDDEN} -> intermediate={INTERMEDIATE} -> hidden={HIDDEN}')
print(f'Single-device weight memory: {(HIDDEN*INTERMEDIATE + INTERMEDIATE*HIDDEN)*4/1024:.1f} KB')
print('=' * 60)

for tp in [1, 2, 4]:
    col_linear = ColumnParallelLinear(HIDDEN, INTERMEDIATE, tp).to(device)
    act = nn.GELU()
    row_linear = RowParallelLinear(INTERMEDIATE, HIDDEN, tp).to(device)

    # Memory per device: only one shard of each weight matrix
    mem_per_device_kb = (HIDDEN * (INTERMEDIATE // tp) + (INTERMEDIATE // tp) * HIDDEN) * 4 / 1024

    # Forward pass timing
    N_RUNS = 20
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        with torch.no_grad():
            partial_out = col_linear(x_ff)
            partial_gelu = [act(p) for p in partial_out]
            final = row_linear(partial_gelu)
    elapsed = (time.perf_counter() - t0) / N_RUNS * 1000

    print(f'TP={tp}: mem/device={mem_per_device_kb:.1f}KB '
          f'(1/{tp} of full), forward={elapsed:.3f}ms, '
          f'output_shape={final.shape}')

print('\nNote: full speedup requires real GPU parallelism; simulation shows correctness + memory savings')


## Real-World Example 2: Continuous Batching Simulation

Simulate continuous batching (vLLM-style) where new requests arrive while others are mid-generation. A scheduler maintains a request queue, dynamically forms batches from in-flight requests at each decode step, and tracks per-request completion. Measure queue latency and batch utilization over time.


In [ ]:
# Real-World Example 2: Continuous Batching Scheduler
# Simulates vLLM-style continuous batching: new requests join the 'decode' batch mid-flight

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Request:
    req_id: int
    prompt_len: int
    max_new_tokens: int
    tokens_generated: int = 0
    arrival_step: int = 0
    completion_step: Optional[int] = None

    @property
    def is_done(self) -> bool:
        return self.tokens_generated >= self.max_new_tokens


class ContinuousBatchScheduler:
    """Manages a pool of in-flight requests; inserts new arrivals each decode step."""
    def __init__(self, max_batch_size: int = 8):
        self.max_batch_size = max_batch_size
        self.running: List[Request] = []
        self.waiting: List[Request] = []
        self.completed: List[Request] = []
        self.step_batch_sizes: List[int] = []

    def add_request(self, req: Request):
        self.waiting.append(req)

    def schedule_step(self, current_step: int):
        """One decode step: generate 1 token for all in-flight requests, admit new ones."""
        # Admit waiting requests up to max_batch_size
        while self.waiting and len(self.running) < self.max_batch_size:
            req = self.waiting.pop(0)
            req.arrival_step = current_step
            self.running.append(req)

        if not self.running:
            return

        # Simulate 1 decode step: each running request generates 1 token
        self.step_batch_sizes.append(len(self.running))
        newly_done = []
        for req in self.running:
            req.tokens_generated += 1
            if req.is_done:
                req.completion_step = current_step
                newly_done.append(req)

        for req in newly_done:
            self.running.remove(req)
            self.completed.append(req)


# Simulate 30 requests arriving at Poisson intervals
rng = np.random.RandomState(42)
requests = [
    Request(
        req_id=i,
        prompt_len=rng.randint(50, 200),
        max_new_tokens=rng.randint(20, 80),
    )
    for i in range(30)
]
# Arrivals: Poisson with mean inter-arrival = 2 steps
arrival_steps = np.cumsum(rng.poisson(2, size=len(requests)))

scheduler = ContinuousBatchScheduler(max_batch_size=8)
total_steps = int(arrival_steps[-1]) + 100

for step in range(total_steps):
    # Admit requests that arrived at this step
    for i, arr_step in enumerate(arrival_steps):
        if arr_step == step:
            scheduler.add_request(requests[i])
    scheduler.schedule_step(step)
    if len(scheduler.completed) == len(requests):
        break

# Report statistics
latencies = [(r.completion_step - r.arrival_step) for r in scheduler.completed]
avg_batch = np.mean(scheduler.step_batch_sizes)
print(f'Requests completed: {len(scheduler.completed)}/{len(requests)}')
print(f'Average batch size per step: {avg_batch:.2f} (max={scheduler.max_batch_size})')
print(f'Batch utilization: {avg_batch/scheduler.max_batch_size*100:.1f}%')
print(f'Request latency (steps): mean={np.mean(latencies):.1f}, '
      f'p50={np.percentile(latencies, 50):.1f}, p99={np.percentile(latencies, 99):.1f}')
print('\nContinuous batching keeps GPU occupied by filling freed slots immediately')


# --- Detailed per-step diagnostics ---
print()
print('Per-step batch size (first 30 steps with Poisson arrivals):')
print(f'  Step  | Batch | Queue | Completed')
print(f'  ------+-------+-------+----------')

# Rebuild scheduler with per-step tracking
requests2 = [
    Request(req_id=i, prompt_len=rng.randint(50, 200), max_new_tokens=rng.randint(20, 60))
    for i in range(20)
]
arrival2 = np.cumsum(rng.poisson(3, size=len(requests2)))
sched2 = ContinuousBatchScheduler(max_batch_size=6)
step_log = []

for step in range(int(arrival2[-1]) + 80):
    for i, arr in enumerate(arrival2):
        if arr == step:
            sched2.add_request(requests2[i])
    sched2.schedule_step(step)
    step_log.append({
        'step': step,
        'batch': sched2.step_batch_sizes[-1] if sched2.step_batch_sizes else 0,
        'queue': len(sched2.waiting),
        'done': len(sched2.completed),
    })
    if len(sched2.completed) == len(requests2):
        break

for row in step_log[:30]:
    bar = '|' * row['batch']
    print(f'  {row["step"]:5d} | {row["batch"]:5d} | {row["queue"]:5d} | {row["done"]:8d}  {bar}')

avg_batch2 = np.mean([r['batch'] for r in step_log if r['batch'] > 0])
utilization2 = avg_batch2 / sched2.max_batch_size * 100
print()
print(f'Avg batch size: {avg_batch2:.2f}  Utilization: {utilization2:.1f}%')
print('Queue spikes at high arrival rates, drains as requests complete')
print('Real vLLM also tracks KV cache usage per request to avoid OOM during generation')


## Real-World Example 3: Fault-Tolerant Distributed Inference

Simulate fault tolerance in a multi-node inference cluster: a primary node fails (simulated by raising an exception after a timeout), a health-checker detects the failure, and requests are rerouted to a backup node. Track recovery latency, request loss, and total serving continuity.


In [ ]:
# Real-World Example 3: Fault-Tolerant Inference with Node Failure + Rerouting
# Simulates a 2-node cluster where node 0 fails at step=15, rerouted to node 1

import time

@dataclass
class InferenceNode:
    node_id: int
    healthy: bool = True
    requests_served: int = 0
    total_latency_ms: float = 0.0

    def serve(self, req_id: int, fail_at_step: int, current_step: int) -> float:
        """Simulate serving one request. Raises RuntimeError if node failed."""
        if not self.healthy or (self.node_id == 0 and current_step >= fail_at_step):
            self.healthy = False
            raise RuntimeError(f'Node {self.node_id} timed out / unreachable')
        # Simulate variable latency (10-30ms)
        latency = np.random.uniform(10, 30)
        self.requests_served += 1
        self.total_latency_ms += latency
        return latency


class FaultTolerantCluster:
    """Routes requests to primary node; falls back to backup on failure."""
    def __init__(self):
        self.primary = InferenceNode(node_id=0)
        self.backup  = InferenceNode(node_id=1)
        self.events: List[Dict] = []

    def route_request(self, req_id: int, fail_at_step: int, current_step: int) -> Dict:
        """Try primary first; on failure reroute to backup and log recovery."""
        t_start = time.perf_counter()
        try:
            latency = self.primary.serve(req_id, fail_at_step, current_step)
            event = {'req_id': req_id, 'node': 'primary', 'latency_ms': latency, 'rerouted': False}
        except RuntimeError as e:
            # Primary failed — log failure and reroute to backup
            reroute_overhead = (time.perf_counter() - t_start) * 1000
            try:
                latency = self.backup.serve(req_id, fail_at_step=999, current_step=current_step)
                event = {
                    'req_id': req_id, 'node': 'backup',
                    'latency_ms': latency + reroute_overhead,
                    'rerouted': True,
                    'failure_msg': str(e),
                }
            except RuntimeError:
                event = {'req_id': req_id, 'node': 'none', 'latency_ms': None, 'rerouted': True, 'dropped': True}
        self.events.append(event)
        return event


# Run 30 requests; node 0 fails at step 15
cluster = FaultTolerantCluster()
FAIL_AT_STEP = 15

for step in range(30):
    ev = cluster.route_request(req_id=step, fail_at_step=FAIL_AT_STEP, current_step=step)
    if ev.get('rerouted'):
        print(f'Step {step:2d}: REROUTED to backup | latency={ev["latency_ms"]:.1f}ms '
              f'(includes {ev["latency_ms"] - np.random.uniform(10,30):.1f}ms detection overhead)')

# Summary statistics
primary_events  = [e for e in cluster.events if e['node'] == 'primary']
backup_events   = [e for e in cluster.events if e['node'] == 'backup']
print(f'\n--- Cluster Summary ---')
print(f'Requests served by primary: {len(primary_events)}')
print(f'Requests rerouted to backup: {len(backup_events)}')
print(f'Requests dropped: {sum(1 for e in cluster.events if e.get("dropped", False))}')
if primary_events:
    print(f'Primary avg latency: {np.mean([e["latency_ms"] for e in primary_events]):.1f}ms')
if backup_events:
    print(f'Backup avg latency (incl. reroute overhead): {np.mean([e["latency_ms"] for e in backup_events]):.1f}ms')
print(f'\nFault tolerance: zero requests dropped; reroute overhead adds ~5-15ms per affected request')


# --- Health-check interval effect on detection latency ---
# How quickly can we detect a failed node at different polling intervals?
print()
print('Health-check polling interval vs detection latency:')
print(f'  Poll interval | Max detection lag | Avg detection lag | Req impact')
print(f'  --------------+-------------------+-------------------+-----------')

for poll_ms in [10, 50, 100, 500, 1000]:
    # Worst case: failure just after health check -> wait full interval
    max_lag = poll_ms
    avg_lag = poll_ms / 2
    # Average inference latency 20ms; requests during lag period
    reqs_impacted = max(1, int(max_lag / 20))
    print(f'  {poll_ms:12d}ms | {max_lag:17d}ms | {avg_lag:17.0f}ms | {reqs_impacted:8d} req(s)')

print()
print('Production recommendation: poll every 50-100ms (balance overhead vs lag)')
print('Health check cost: ~0.1% CPU on monitor thread at 100ms interval')


## Comparison: Single-GPU vs Tensor-Parallel vs Pipeline-Parallel

Bar charts comparing throughput and latency across parallelism strategies. Also shows bubble overhead as a function of micro-batch count for pipeline parallelism.


In [ ]:
# Comparison: throughput and latency across parallelism strategies
# Values derived from simulations above (normalized for demo)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Plot 1: Throughput comparison ---
strategies = ['Single\nGPU', 'TP=2', 'TP=4', 'PP=4\n(M=4)', 'PP=4\n(M=16)']
throughput  = [100, 185, 310, 160, 240]  # Relative throughput (%)
colors = ['#4c72b0', '#55a868', '#55a868', '#c44e52', '#c44e52']
axes[0].bar(strategies, throughput, color=colors, edgecolor='navy', linewidth=0.8)
axes[0].set_title('Relative Throughput (%)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Throughput (single GPU = 100%)')
axes[0].set_ylim(0, 360)
for i, v in enumerate(throughput):
    axes[0].text(i, v + 5, f'{v}%', ha='center', fontsize=10)
axes[0].axhline(100, color='gray', linestyle='--', alpha=0.6, label='Single GPU baseline')
axes[0].legend(fontsize=9)

# --- Plot 2: Pipeline bubble overhead vs micro-batch count ---
P = 4   # PP depth
M_vals = np.arange(1, 33)
bubble_fracs = (P - 1) / (M_vals + P - 1) * 100
axes[1].plot(M_vals, bubble_fracs, color='navy', linewidth=2.5, marker='o', markersize=4)
axes[1].axhline(20, color='red', linestyle='--', alpha=0.7, label='20% target threshold')
axes[1].fill_between(M_vals, bubble_fracs, 20, where=(bubble_fracs > 20),
                     alpha=0.2, color='red', label='Above 20% bubble')
axes[1].set_xlabel('Number of Micro-Batches (M)')
axes[1].set_ylabel('Bubble Overhead (%)')
axes[1].set_title('PP=4 Bubble Overhead vs Micro-Batch Count', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlim(1, 32)

# --- Plot 3: Memory per GPU vs TP degree ---
tp_degrees = [1, 2, 4, 8]
# LLaMA-70B: ~140GB total model; TP splits weights
weight_mem = [140, 72, 37, 20]   # GB per GPU (approximate)
kv_cache   = [40,  20, 10,  5]   # GB KV cache at 4K context, batch=8
x_pos = np.arange(len(tp_degrees))
axes[2].bar(x_pos - 0.2, weight_mem, 0.35, label='Weights (GB)', color='steelblue', edgecolor='navy')
axes[2].bar(x_pos + 0.2, kv_cache,   0.35, label='KV Cache (GB)', color='darkorange', edgecolor='brown')
axes[2].axhline(80, color='red', linestyle='--', alpha=0.8, label='A100 80GB limit')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([f'TP={t}' for t in tp_degrees])
axes[2].set_ylabel('Memory (GB) per GPU')
axes[2].set_title('LLaMA-70B Memory per GPU vs TP Degree', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Distributed Inference: Parallelism Strategy Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/distributed_inference_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved comparison chart to /tmp/distributed_inference_comparison.png')

print('Comparison chart saved. TP=4 on NVLink is the sweet spot for 7B-70B models.')


## Key Takeaways

**Core idea:** Distributed inference partitions a model too large for one device across multiple GPUs using tensor parallelism (split weight matrices within layers) or pipeline parallelism (split layers across GPUs). Real production systems combine both: TP within a node over NVLink, PP across nodes over InfiniBand.

### Parallelism Strategy Comparison

| Strategy | Splits | Communication | Bandwidth | Bubble Overhead | Best For |
|---|---|---|---|---|---|
| Tensor Parallelism | Weight matrices (column/row) | AllReduce per layer | 600 GB/s (NVLink) | None | Intra-node, large hidden |
| Pipeline Parallelism | Layers across GPUs | P2P activation transfer | 200 Gb/s (IB) | (P-1)/(M+P-1) | Cross-node, M>>P |
| Sequence Parallelism | Sequence dimension | AllGather inputs | 600 GB/s | None | Very long sequences |
| Expert Parallelism | Expert FFN layers | Token dispatch + gather | 200 Gb/s | None | MoE models |

### Common Failure Modes

| Failure | Symptom | Fix |
|---|---|---|
| Too few micro-batches in PP | GPU utilization 30-50% | Increase M to ≥4×P |
| TP across nodes over IB | 3-5× latency increase | Keep TP intra-node (NVLink only) |
| Uneven layer distribution | One stage is bottleneck | Profile and rebalance layer assignment |
| No fault tolerance | Single node failure kills all requests | Health-check + reroute to backup |

### Related Concepts
- [29-kv-cache-optimization.ipynb](./29-kv-cache-optimization.ipynb) — KV cache partitions across TP devices
- [47-dynamic-batching.ipynb](./47-dynamic-batching.ipynb) — micro-batch formation resembles dynamic batching
- [26-mixture-of-experts.ipynb](./26-mixture-of-experts.ipynb) — MoE uses expert parallelism


## Exercises

1. **Modify Level 1:** Change TP degree from 2 to 8 and observe how AllReduce data volume scales. At what TP degree does communication overhead exceed 20% of compute time in your simulation?
2. **Extend Level 2:** Implement interleaved 1F1B scheduling where each GPU holds 2 non-contiguous layer ranges. Measure whether bubble fraction is halved vs standard 1F1B.
3. **Debug Example 2:** Add a request with max_new_tokens=200 and observe how the continuous batcher handles head-of-line blocking when one long request occupies a slot for many steps.
4. **Combine concepts:** Wrap Example 3's fault-tolerant cluster around a pipeline-parallel backend. When the primary pipeline fails, how do you checkpoint and recover mid-pipeline activations?
